In [7]:
# 2차 데이터 개선 1번 셀: previous-day feature dataset 가능성 확인
# 목적:
# - sleep target calendar_date 기준으로 같은 날짜 feature를 쓰지 않고, 전날 feature만 사용할 수 있는지 확인한다.
# - participant별로 feature columns를 1일 shift한다.
# - target 날짜와 feature 날짜가 정확히 하루 차이인지 검증한다.
# - 아직 파일 저장은 하지 않는다.

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"c:\workSpace\DeepLearnin_sleep")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

DATA_PATH = PROCESSED_DIR / "modeling_dataset_encoded.csv"
FEATURE_PATH = PROCESSED_DIR / "encoded_feature_columns.csv"
SPLIT_PATH = PROCESSED_DIR / "deep_learning_sequences" / "deep_learning_participant_split_assignments.csv"

TARGET_COL = "good_sleep_label"
ID_COL = "participant_object_id"
DATE_COL = "calendar_date"
SPLIT_COL = "deep_learning_split"

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
feature_df = pd.read_csv(FEATURE_PATH, encoding="utf-8-sig")
split_df = pd.read_csv(SPLIT_PATH, encoding="utf-8-sig")

feature_cols = feature_df["feature"].tolist()

split_df = split_df[[ID_COL, SPLIT_COL]].drop_duplicates()
df = df.merge(split_df, on=ID_COL, how="left")

df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values([ID_COL, DATE_COL]).reset_index(drop=True)

print("original df shape:", df.shape)
print("feature count:", len(feature_cols))
print(df[SPLIT_COL].value_counts(dropna=False))


def create_previous_day_frame(data, feature_columns):
    base_cols = [ID_COL, DATE_COL, TARGET_COL, SPLIT_COL]

    target_frame = data[base_cols].copy()
    target_frame = target_frame.rename(columns={DATE_COL: "target_date"})

    feature_frame = data[[ID_COL, DATE_COL] + feature_columns].copy()
    feature_frame = feature_frame.rename(columns={DATE_COL: "feature_date"})

    feature_frame["target_date"] = feature_frame["feature_date"] + pd.Timedelta(days=1)

    previous_day = target_frame.merge(
        feature_frame,
        on=[ID_COL, "target_date"],
        how="left",
    )

    previous_day["feature_lag_days"] = (
        previous_day["target_date"] - previous_day["feature_date"]
    ).dt.days

    return previous_day


previous_day_df = create_previous_day_frame(df, feature_cols)

coverage_summary = {
    "original_rows": len(df),
    "previous_day_rows": len(previous_day_df),
    "rows_with_previous_day_features": int(previous_day_df["feature_date"].notna().sum()),
    "rows_without_previous_day_features": int(previous_day_df["feature_date"].isna().sum()),
    "previous_day_coverage_rate": float(previous_day_df["feature_date"].notna().mean()),
}

print('coverage_summary')
print(coverage_summary)

lag_counts = previous_day_df["feature_lag_days"].value_counts(dropna=False).sort_index()
print("lag_counts")
display(lag_counts)

split_coverage = (
    previous_day_df
    .groupby(SPLIT_COL)
    .agg(
        rows=("target_date", "size"),
        rows_with_previous_day_features=("feature_date", lambda s: int(s.notna().sum())),
        coverage_rate=("feature_date", lambda s: float(s.notna().mean())),
        participants=(ID_COL, "nunique"),
        target_mean=(TARGET_COL, "mean"),
    )
    .reset_index()
)

display(split_coverage)

usable_previous_day_df = previous_day_df[previous_day_df["feature_date"].notna()].copy()

print("usable_previous_day_df shape:", usable_previous_day_df.shape)
print("usable split counts:")
print(usable_previous_day_df[SPLIT_COL].value_counts())

participant_coverage = (
    previous_day_df
    .assign(has_previous_day=previous_day_df["feature_date"].notna())
    .groupby([SPLIT_COL, ID_COL], as_index=False)
    .agg(
        rows=("target_date", "size"),
        usable_rows=("has_previous_day", "sum"),
        coverage_rate=("has_previous_day", "mean"),
        target_mean=(TARGET_COL, "mean"),
    )
)

display(
    participant_coverage
    .groupby(SPLIT_COL)
    .agg(
        participants=(ID_COL, "nunique"),
        mean_rows=("rows", "mean"),
        mean_usable_rows=("usable_rows", "mean"),
        min_usable_rows=("usable_rows", "min"),
        max_usable_rows=("usable_rows", "max"),
        mean_coverage_rate=("coverage_rate", "mean"),
    )
    .reset_index()
)

original df shape: (3551, 201)
feature count: 197
deep_learning_split
train         2425
test           607
validation     519
Name: count, dtype: int64
coverage_summary
{'original_rows': 3551, 'previous_day_rows': 3551, 'rows_with_previous_day_features': 3116, 'rows_without_previous_day_features': 435, 'previous_day_coverage_rate': 0.8774992959729654}
lag_counts


feature_lag_days
1.0    3116
NaN     435
Name: count, dtype: int64

,deep_learning_split,rows,rows_with_previous_day_features,coverage_rate,participants,target_mean
0,test,607,524,0.863262,14,0.369028
1,train,2425,2135,0.880412,44,0.400000
2,validation,519,457,0.880539,11,0.393064


usable_previous_day_df shape: (3116, 203)
usable split counts:
deep_learning_split
train         2135
test           524
validation     457
Name: count, dtype: int64


,deep_learning_split,participants,mean_rows,mean_usable_rows,min_usable_rows,max_usable_rows,mean_coverage_rate
0,test,14,43.357143,37.428571,0,79,0.697417
1,train,44,55.113636,48.522727,0,200,0.780599
2,validation,11,47.181818,41.545455,0,81,0.752945


In [8]:
# 2차 데이터 개선 2번 셀: previous-day modeling dataset 저장
# 목적:
# - target_date의 label은 그대로 두고, feature는 target_date - 1일 값만 사용한다.
# - same-date feature 사용 위험을 줄인 previous-day dataset을 만든다.
# - feature column 이름은 기존과 동일하게 유지한다.
# - feature_date, feature_lag_days를 함께 저장해서 추적 가능하게 한다.

PREVIOUS_DAY_DIR = PROCESSED_DIR / "deep_learning_previous_day"
PREVIOUS_DAY_DIR.mkdir(parents=True, exist_ok=True)

previous_day_usable = previous_day_df[previous_day_df["feature_date"].notna()].copy()

previous_day_usable = previous_day_usable.rename(columns={"target_date": DATE_COL})

ordered_cols = [
    ID_COL,
    DATE_COL,
    "feature_date",
    "feature_lag_days",
    TARGET_COL,
    SPLIT_COL,
] + feature_cols

previous_day_usable = previous_day_usable[ordered_cols].sort_values(
    [ID_COL, DATE_COL]
).reset_index(drop=True)

previous_day_dataset_path = PREVIOUS_DAY_DIR / "modeling_dataset_previous_day_encoded.csv"
previous_day_feature_path = PREVIOUS_DAY_DIR / "previous_day_feature_columns.csv"

previous_day_usable.to_csv(
    previous_day_dataset_path,
    index=False,
    encoding="utf-8-sig",
)

pd.DataFrame({"feature": feature_cols}).to_csv(
    previous_day_feature_path,
    index=False,
    encoding="utf-8-sig",
)

previous_day_summary = {
    "path": str(previous_day_dataset_path),
    "rows": len(previous_day_usable),
    "features": len(feature_cols),
    "participants": previous_day_usable[ID_COL].nunique(),
    "min_target_date": str(previous_day_usable[DATE_COL].min().date()),
    "max_target_date": str(previous_day_usable[DATE_COL].max().date()),
    "feature_lag_days_unique": sorted(previous_day_usable["feature_lag_days"].dropna().unique().tolist()),
}

print(previous_day_summary)

display(previous_day_usable[SPLIT_COL].value_counts().rename("rows"))

display(
    previous_day_usable
    .groupby(SPLIT_COL)
    .agg(
        rows=(TARGET_COL, "size"),
        participants=(ID_COL, "nunique"),
        target_mean=(TARGET_COL, "mean"),
        min_target_date=(DATE_COL, "min"),
        max_target_date=(DATE_COL, "max"),
    )
    .reset_index()
)

{'path': 'c:\\workSpace\\DeepLearnin_sleep\\data\\processed\\deep_learning_previous_day\\modeling_dataset_previous_day_encoded.csv', 'rows': 3116, 'features': 197, 'participants': 63, 'min_target_date': '2021-05-25', 'max_target_date': '2022-01-22', 'feature_lag_days_unique': [1.0]}


deep_learning_split
train         2135
test           524
validation     457
Name: rows, dtype: int64

,deep_learning_split,rows,participants,target_mean,min_target_date,max_target_date
0,test,524,12,0.379771,2021-05-25,2022-01-22
1,train,2135,41,0.405621,2021-05-25,2022-01-21
2,validation,457,10,0.393873,2021-05-25,2022-01-21


In [9]:
# 2차 데이터 개선 3번 셀: previous-day feature subset별 sequence tensor 생성
# 목적:
# - previous-day encoded dataset을 사용해서 subset별 sequence tensor를 만든다.
# - 기존 Phase 1A와 같은 subset 정의를 재사용한다.
# - 출력은 deep_learning_previous_day/sequences_by_subset 아래에 저장한다.

from sklearn.preprocessing import StandardScaler
import joblib
import json

PREVIOUS_DAY_DIR = PROCESSED_DIR / "deep_learning_previous_day"
PREVIOUS_DAY_DATA_PATH = PREVIOUS_DAY_DIR / "modeling_dataset_previous_day_encoded.csv"

SUBSET_DIR = PROCESSED_DIR / "deep_learning_feature_subsets"
PREVIOUS_DAY_SEQUENCE_ROOT = PREVIOUS_DAY_DIR / "sequences_by_subset"

WINDOW_SIZES = [7, 14, 30]

previous_df = pd.read_csv(PREVIOUS_DAY_DATA_PATH, encoding="utf-8-sig")
previous_df[DATE_COL] = pd.to_datetime(previous_df[DATE_COL])
previous_df["feature_date"] = pd.to_datetime(previous_df["feature_date"])
previous_df = previous_df.sort_values([ID_COL, DATE_COL]).reset_index(drop=True)

subset_paths = sorted(SUBSET_DIR.glob("*_features.csv"))

feature_subsets = {}

for path in subset_paths:
    subset_name = path.name.replace("_features.csv", "")
    subset_df = pd.read_csv(path, encoding="utf-8-sig")
    feature_subsets[subset_name] = subset_df["feature"].tolist()

print("previous_df shape:", previous_df.shape)
print("subsets:", {k: len(v) for k, v in feature_subsets.items()})


def scale_previous_subset(data, features):
    scaled = data.copy()
    train_mask = scaled[SPLIT_COL] == "train"

    scaler = StandardScaler()
    scaler.fit(scaled.loc[train_mask, features])

    scaled.loc[:, features] = scaler.transform(scaled[features]).astype(np.float32)

    return scaled, scaler


def build_previous_windows_for_split(data, features, split_name, window_size):
    split_df = data[data[SPLIT_COL] == split_name].copy()
    split_df = split_df.sort_values([ID_COL, DATE_COL])

    X_sequence = []
    X_mlp_flattened = []
    X_mlp_current_day = []
    y = []
    participant_ids = []
    window_start_dates = []
    window_end_dates = []
    feature_start_dates = []
    feature_end_dates = []

    for participant_id, group in split_df.groupby(ID_COL, sort=False):
        group = group.sort_values(DATE_COL).reset_index(drop=True)

        feature_values = group[features].to_numpy(dtype=np.float32)
        target_values = group[TARGET_COL].to_numpy(dtype=np.int64)
        target_dates = group[DATE_COL].to_numpy()
        feature_dates = group["feature_date"].to_numpy()

        for end_idx in range(window_size - 1, len(group)):
            start_idx = end_idx - window_size + 1

            target_window_dates = pd.to_datetime(target_dates[start_idx : end_idx + 1])
            feature_window_dates = pd.to_datetime(feature_dates[start_idx : end_idx + 1])

            expected_target_dates = pd.date_range(
                start=target_window_dates[0],
                periods=window_size,
                freq="D",
            )

            expected_feature_dates = expected_target_dates - pd.Timedelta(days=1)

            target_contiguous = np.array_equal(
                target_window_dates.to_numpy(),
                expected_target_dates.to_numpy(),
            )

            feature_contiguous = np.array_equal(
                feature_window_dates.to_numpy(),
                expected_feature_dates.to_numpy(),
            )

            if not target_contiguous or not feature_contiguous:
                continue

            window_x = feature_values[start_idx : end_idx + 1]

            X_sequence.append(window_x)
            X_mlp_flattened.append(window_x.reshape(-1))
            X_mlp_current_day.append(window_x[-1])
            y.append(target_values[end_idx])
            participant_ids.append(participant_id)
            window_start_dates.append(str(pd.Timestamp(target_window_dates[0]).date()))
            window_end_dates.append(str(pd.Timestamp(target_window_dates[-1]).date()))
            feature_start_dates.append(str(pd.Timestamp(feature_window_dates[0]).date()))
            feature_end_dates.append(str(pd.Timestamp(feature_window_dates[-1]).date()))

    if X_sequence:
        X_sequence = np.stack(X_sequence).astype(np.float32)
        X_mlp_flattened = np.stack(X_mlp_flattened).astype(np.float32)
        X_mlp_current_day = np.stack(X_mlp_current_day).astype(np.float32)
        y = np.asarray(y, dtype=np.int64)
    else:
        X_sequence = np.empty((0, window_size, len(features)), dtype=np.float32)
        X_mlp_flattened = np.empty((0, window_size * len(features)), dtype=np.float32)
        X_mlp_current_day = np.empty((0, len(features)), dtype=np.float32)
        y = np.empty((0,), dtype=np.int64)

    return {
        "X_sequence": X_sequence,
        "X_mlp_flattened": X_mlp_flattened,
        "X_mlp_current_day": X_mlp_current_day,
        "y": y,
        "participant_object_id": np.asarray(participant_ids, dtype=object),
        "window_start_date": np.asarray(window_start_dates, dtype=object),
        "window_end_date": np.asarray(window_end_dates, dtype=object),
        "feature_start_date": np.asarray(feature_start_dates, dtype=object),
        "feature_end_date": np.asarray(feature_end_dates, dtype=object),
        "feature_names": np.asarray(features, dtype=object),
    }


summary_rows = []

PREVIOUS_DAY_SEQUENCE_ROOT.mkdir(parents=True, exist_ok=True)

for subset_name, features in feature_subsets.items():
    print(f"\n=== subset: {subset_name} ({len(features)} features) ===")

    scaled_df, scaler = scale_previous_subset(previous_df, features)

    subset_root = PREVIOUS_DAY_SEQUENCE_ROOT / subset_name
    subset_root.mkdir(parents=True, exist_ok=True)

    joblib.dump(scaler, subset_root / "standard_scaler.joblib")

    pd.DataFrame({"feature": features}).to_csv(
        subset_root / "feature_columns.csv",
        index=False,
        encoding="utf-8-sig",
    )

    for window_size in WINDOW_SIZES:
        window_root = subset_root / f"window_{window_size}"
        window_root.mkdir(parents=True, exist_ok=True)

        all_sample_index = []

        for split_name in ["train", "validation", "test"]:
            arrays = build_previous_windows_for_split(
                scaled_df,
                features,
                split_name,
                window_size,
            )

            npz_path = window_root / f"{split_name}.npz"
            np.savez_compressed(npz_path, **arrays)

            sample_index = pd.DataFrame({
                "window": window_size,
                "split": split_name,
                "participant_object_id": arrays["participant_object_id"],
                "window_start_date": arrays["window_start_date"],
                "window_end_date": arrays["window_end_date"],
                "feature_start_date": arrays["feature_start_date"],
                "feature_end_date": arrays["feature_end_date"],
                "good_sleep_label": arrays["y"],
            })

            all_sample_index.append(sample_index)

            y = arrays["y"]

            summary_rows.append({
                "subset_name": subset_name,
                "window": window_size,
                "split": split_name,
                "samples": int(len(y)),
                "participants": int(len(set(arrays["participant_object_id"]))),
                "features": int(len(features)),
                "sequence_shape": " x ".join(map(str, arrays["X_sequence"].shape)),
                "mlp_flattened_shape": " x ".join(map(str, arrays["X_mlp_flattened"].shape)),
                "mlp_current_day_shape": " x ".join(map(str, arrays["X_mlp_current_day"].shape)),
                "class_0_samples": int((y == 0).sum()),
                "class_1_samples": int((y == 1).sum()),
                "good_sleep_label_mean": float(y.mean()) if len(y) else np.nan,
                "tensor_path": str(npz_path),
            })

            print(
                f"window={window_size:>2} split={split_name:<10} "
                f"samples={len(y):>4} shape={arrays['X_sequence'].shape}"
            )

        pd.concat(all_sample_index, ignore_index=True).to_csv(
            window_root / "sample_index.csv",
            index=False,
            encoding="utf-8-sig",
        )

    metadata = {
        "subset_name": subset_name,
        "feature_count": len(features),
        "windows": WINDOW_SIZES,
        "target_col": TARGET_COL,
        "id_col": ID_COL,
        "date_col": DATE_COL,
        "split_col": SPLIT_COL,
        "source_data": str(PREVIOUS_DAY_DATA_PATH),
        "feature_policy": "target_date uses features from target_date_minus_1",
    }

    with open(subset_root / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

previous_sequence_summary = pd.DataFrame(summary_rows)

previous_sequence_summary_path = PREVIOUS_DAY_SEQUENCE_ROOT / "sequence_tensor_summary_previous_day_by_subset.csv"
previous_sequence_summary.to_csv(
    previous_sequence_summary_path,
    index=False,
    encoding="utf-8-sig",
)

print("\nsaved:", previous_sequence_summary_path)
display(previous_sequence_summary)

previous_df shape: (3116, 203)
subsets: {'daytime_context_survey': 115, 'daytime_only': 19, 'full_current': 197, 'no_high_sleep_session': 127, 'no_high_sleep_session_no_stress': 115}

=== subset: daytime_context_survey (115 features) ===
window= 7 split=train      samples=1303 shape=(1303, 7, 115)
window= 7 split=validation samples= 245 shape=(245, 7, 115)
window= 7 split=test       samples= 296 shape=(296, 7, 115)
window=14 split=train      samples= 887 shape=(887, 14, 115)
window=14 split=validation samples= 136 shape=(136, 14, 115)
window=14 split=test       samples= 206 shape=(206, 14, 115)
window=30 split=train      samples= 415 shape=(415, 30, 115)
window=30 split=validation samples=  52 shape=(52, 30, 115)
window=30 split=test       samples= 101 shape=(101, 30, 115)

=== subset: daytime_only (19 features) ===
window= 7 split=train      samples=1303 shape=(1303, 7, 19)
window= 7 split=validation samples= 245 shape=(245, 7, 19)
window= 7 split=test       samples= 296 shape=(296, 7

,subset_name,window,split,samples,participants,features,sequence_shape,mlp_flattened_shape,mlp_current_day_shape,class_0_samples,class_1_samples,good_sleep_label_mean,tensor_path
0,daytime_context_survey,7,train,1303,33,115,1303 x 7 x 115,1303 x 805,1303 x 115,779,524,0.402149,c:\workSpace\DeepLearnin_sleep\data\processed\...
1,daytime_context_survey,7,validation,245,9,115,245 x 7 x 115,245 x 805,245 x 115,153,92,0.375510,c:\workSpace\DeepLearnin_sleep\data\processed\...
2,daytime_context_survey,7,test,296,9,115,296 x 7 x 115,296 x 805,296 x 115,173,123,0.415541,c:\workSpace\DeepLearnin_sleep\data\processed\...
3,daytime_context_survey,14,train,887,27,115,887 x 14 x 115,887 x 1610,887 x 115,553,334,0.376550,c:\workSpace\DeepLearnin_sleep\data\processed\...
4,daytime_context_survey,14,validation,136,7,115,136 x 14 x 115,136 x 1610,136 x 115,88,48,0.352941,c:\workSpace\DeepLearnin_sleep\data\processed\...
5,daytime_context_survey,14,test,206,5,115,206 x 14 x 115,206 x 1610,206 x 115,109,97,0.470874,c:\workSpace\DeepLearnin_sleep\data\processed\...
6,daytime_context_survey,30,train,415,17,115,415 x 30 x 115,415 x 3450,415 x 115,271,144,0.346988,c:\workSpace\DeepLearnin_sleep\data\processed\...
7,daytime_context_survey,30,validation,52,3,115,52 x 30 x 115,52 x 3450,52 x 115,40,12,0.230769,c:\workSpace\DeepLearnin_sleep\data\processed\...
8,daytime_context_survey,30,test,101,4,115,101 x 30 x 115,101 x 3450,101 x 115,41,60,0.594059,c:\workSpace\DeepLearnin_sleep\data\processed\...
9,daytime_only,7,train,1303,33,19,1303 x 7 x 19,1303 x 133,1303 x 19,779,524,0.402149,c:\workSpace\DeepLearnin_sleep\data\processed\...


In [ ]:
# 2차 데이터 개선 4번 셀: previous-day tensor 검증 + Phase 2A grid 생성
# 목적:
# - previous-day sequence tensor 파일들이 정상인지 검증한다.
# - 같은-date Phase 1A에서 유망했던 조합 중심으로 previous-day Phase 2A grid를 만든다.
# - 우선 daytime_only 중심, window 7/14 중심으로 간다.

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"c:\workSpace\DeepLearnin_sleep")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PREVIOUS_DAY_DIR = PROCESSED_DIR / "deep_learning_previous_day"
PREVIOUS_DAY_SEQUENCE_ROOT = PREVIOUS_DAY_DIR / "sequences_by_subset"
PREVIOUS_DAY_SUMMARY_PATH = (
    PREVIOUS_DAY_SEQUENCE_ROOT / "sequence_tensor_summary_previous_day_by_subset.csv"
)

EXPERIMENT_DIR = PREVIOUS_DAY_DIR / "experiments"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

previous_sequence_summary = pd.read_csv(PREVIOUS_DAY_SUMMARY_PATH, encoding="utf-8-sig")

print("summary shape:", previous_sequence_summary.shape)
display(previous_sequence_summary)

validation_rows = []

for _, row in previous_sequence_summary.iterrows():
    subset_name = row["subset_name"]
    window = int(row["window"])
    split = row["split"]

    subset_root = PREVIOUS_DAY_SEQUENCE_ROOT / subset_name
    window_root = subset_root / f"window_{window}"

    npz_path = window_root / f"{split}.npz"
    feature_path = subset_root / "feature_columns.csv"
    sample_index_path = window_root / "sample_index.csv"

    arrays = np.load(npz_path, allow_pickle=True)
    feature_df = pd.read_csv(feature_path, encoding="utf-8-sig")
    sample_index = pd.read_csv(sample_index_path, encoding="utf-8-sig")

    X_sequence = arrays["X_sequence"]
    X_mlp_current_day = arrays["X_mlp_current_day"]
    X_mlp_flattened = arrays["X_mlp_flattened"]
    y = arrays["y"]
    feature_names = arrays["feature_names"]

    sample_index_split = sample_index[sample_index["split"] == split]

    validation_rows.append({
        "subset_name": subset_name,
        "window": window,
        "split": split,
        "npz_exists": npz_path.exists(),
        "feature_csv_count": len(feature_df),
        "feature_npz_count": len(feature_names),
        "summary_features": int(row["features"]),
        "samples_npz": len(y),
        "samples_summary": int(row["samples"]),
        "samples_index": len(sample_index_split),
        "X_sequence_shape": str(X_sequence.shape),
        "has_nan": bool(np.isnan(X_sequence).any()),
        "has_inf": bool(np.isinf(X_sequence).any()),
        "feature_count_match": len(feature_df) == len(feature_names) == int(row["features"]),
        "sample_count_match": len(y) == int(row["samples"]) == len(sample_index_split),
        "sequence_shape_match": X_sequence.shape == (len(y), window, len(feature_names)),
        "mlp_current_day_shape_match": X_mlp_current_day.shape == (len(y), len(feature_names)),
        "mlp_flattened_shape_match": X_mlp_flattened.shape == (len(y), window * len(feature_names)),
    })

validation_df = pd.DataFrame(validation_rows)

problem_cols = [
    "npz_exists",
    "feature_count_match",
    "sample_count_match",
    "sequence_shape_match",
    "mlp_current_day_shape_match",
    "mlp_flattened_shape_match",
]

problems = validation_df[
    (validation_df[problem_cols] == False).any(axis=1)
    | validation_df["has_nan"]
    | validation_df["has_inf"]
]

print("problem rows:", len(problems))
display(problems)

compact_summary = (
    validation_df
    .groupby(["subset_name", "window"], as_index=False)
    .agg(
        feature_count=("feature_npz_count", "first"),
        train_samples=("samples_npz", lambda s: int(s.iloc[0])),
        all_feature_count_match=("feature_count_match", "all"),
        all_sample_count_match=("sample_count_match", "all"),
        all_sequence_shape_match=("sequence_shape_match", "all"),
        any_nan=("has_nan", "any"),
        any_inf=("has_inf", "any"),
    )
)

display(compact_summary)


# Phase 2A는 previous-day 데이터에서 가장 중요한 조합부터 실행한다.
# Phase 1A 결과상 daytime_only가 가장 안정적이었고, window 7/14가 participant 수 측면에서 더 안전했다.
phase_2a_rows = []

PHASE_2A_SUBSETS = [
    "daytime_only",
    "no_high_sleep_session",
]

PHASE_2A_WINDOWS = [
    7,
    14,
]

PHASE_2A_MODELS = [
    "mlp_current_day",
    "gru",
    "bilstm",
    "cnn_1d",
]

for subset_name in PHASE_2A_SUBSETS:
    for window in PHASE_2A_WINDOWS:
        for model_family in PHASE_2A_MODELS:
            phase_2a_rows.append({
                "phase": "phase_2a_previous_day_priority",
                "subset_name": subset_name,
                "window": window,
                "model_family": model_family,
                "run": True,
                "selection_metric": "validation_balanced_accuracy",
                "test_policy": "validation ranking 이후 확인",
            })

phase_2a_grid = pd.DataFrame(phase_2a_rows)
phase_2a_grid["experiment_id"] = [
    f"phase2a_{i:03d}" for i in range(len(phase_2a_grid))
]

phase_2a_grid = phase_2a_grid[
    [
        "experiment_id",
        "phase",
        "subset_name",
        "window",
        "model_family",
        "run",
        "selection_metric",
        "test_policy",
    ]
]

phase_2a_grid_path = EXPERIMENT_DIR / "phase_2a_previous_day_experiment_grid.csv"
phase_2a_grid.to_csv(phase_2a_grid_path, index=False, encoding="utf-8-sig")

print("saved:", phase_2a_grid_path)
print("phase_2a experiments:", len(phase_2a_grid))

display(phase_2a_grid)

display(
    phase_2a_grid
    .groupby(["subset_name", "window"], as_index=False)
    .agg(
        experiments=("experiment_id", "count"),
        models=("model_family", lambda s: sorted(s.unique())),
    )
)

summary shape: (45, 13)


,subset_name,window,split,samples,participants,features,sequence_shape,mlp_flattened_shape,mlp_current_day_shape,class_0_samples,class_1_samples,good_sleep_label_mean,tensor_path
0,daytime_context_survey,7,train,1303,33,115,1303 x 7 x 115,1303 x 805,1303 x 115,779,524,0.402149,c:\workSpace\DeepLearnin_sleep\data\processed\...
1,daytime_context_survey,7,validation,245,9,115,245 x 7 x 115,245 x 805,245 x 115,153,92,0.375510,c:\workSpace\DeepLearnin_sleep\data\processed\...
2,daytime_context_survey,7,test,296,9,115,296 x 7 x 115,296 x 805,296 x 115,173,123,0.415541,c:\workSpace\DeepLearnin_sleep\data\processed\...
3,daytime_context_survey,14,train,887,27,115,887 x 14 x 115,887 x 1610,887 x 115,553,334,0.376550,c:\workSpace\DeepLearnin_sleep\data\processed\...
4,daytime_context_survey,14,validation,136,7,115,136 x 14 x 115,136 x 1610,136 x 115,88,48,0.352941,c:\workSpace\DeepLearnin_sleep\data\processed\...
5,daytime_context_survey,14,test,206,5,115,206 x 14 x 115,206 x 1610,206 x 115,109,97,0.470874,c:\workSpace\DeepLearnin_sleep\data\processed\...
6,daytime_context_survey,30,train,415,17,115,415 x 30 x 115,415 x 3450,415 x 115,271,144,0.346988,c:\workSpace\DeepLearnin_sleep\data\processed\...
7,daytime_context_survey,30,validation,52,3,115,52 x 30 x 115,52 x 3450,52 x 115,40,12,0.230769,c:\workSpace\DeepLearnin_sleep\data\processed\...
8,daytime_context_survey,30,test,101,4,115,101 x 30 x 115,101 x 3450,101 x 115,41,60,0.594059,c:\workSpace\DeepLearnin_sleep\data\processed\...
9,daytime_only,7,train,1303,33,19,1303 x 7 x 19,1303 x 133,1303 x 19,779,524,0.402149,c:\workSpace\DeepLearnin_sleep\data\processed\...


problem rows: 0


,subset_name,window,split,npz_exists,feature_csv_count,feature_npz_count,summary_features,samples_npz,samples_summary,samples_index,X_sequence_shape,has_nan,has_inf,feature_count_match,sample_count_match,sequence_shape_match,mlp_current_day_shape_match,mlp_flattened_shape_match


,subset_name,window,feature_count,train_samples,all_feature_count_match,all_sample_count_match,all_sequence_shape_match,any_nan,any_inf
0,daytime_context_survey,7,115,1303,True,True,True,False,False
1,daytime_context_survey,14,115,887,True,True,True,False,False
2,daytime_context_survey,30,115,415,True,True,True,False,False
3,daytime_only,7,19,1303,True,True,True,False,False
4,daytime_only,14,19,887,True,True,True,False,False
5,daytime_only,30,19,415,True,True,True,False,False
6,full_current,7,197,1303,True,True,True,False,False
7,full_current,14,197,887,True,True,True,False,False
8,full_current,30,197,415,True,True,True,False,False
9,no_high_sleep_session,7,127,1303,True,True,True,False,False


saved: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_previous_day\experiments\phase_2a_previous_day_experiment_grid.csv
phase_2a experiments: 16


,experiment_id,phase,subset_name,window,model_family,run,selection_metric,test_policy
0,phase2a_000,phase_2a_previous_day_priority,daytime_only,7,mlp_current_day,True,validation_balanced_accuracy,validation ranking 이후 확인
1,phase2a_001,phase_2a_previous_day_priority,daytime_only,7,gru,True,validation_balanced_accuracy,validation ranking 이후 확인
2,phase2a_002,phase_2a_previous_day_priority,daytime_only,7,bilstm,True,validation_balanced_accuracy,validation ranking 이후 확인
3,phase2a_003,phase_2a_previous_day_priority,daytime_only,7,cnn_1d,True,validation_balanced_accuracy,validation ranking 이후 확인
4,phase2a_004,phase_2a_previous_day_priority,daytime_only,14,mlp_current_day,True,validation_balanced_accuracy,validation ranking 이후 확인
5,phase2a_005,phase_2a_previous_day_priority,daytime_only,14,gru,True,validation_balanced_accuracy,validation ranking 이후 확인
6,phase2a_006,phase_2a_previous_day_priority,daytime_only,14,bilstm,True,validation_balanced_accuracy,validation ranking 이후 확인
7,phase2a_007,phase_2a_previous_day_priority,daytime_only,14,cnn_1d,True,validation_balanced_accuracy,validation ranking 이후 확인
8,phase2a_008,phase_2a_previous_day_priority,no_high_sleep_session,7,mlp_current_day,True,validation_balanced_accuracy,validation ranking 이후 확인
9,phase2a_009,phase_2a_previous_day_priority,no_high_sleep_session,7,gru,True,validation_balanced_accuracy,validation ranking 이후 확인


,subset_name,window,experiments,models
0,daytime_only,7,4,"[bilstm, cnn_1d, gru, mlp_current_day]"
1,daytime_only,14,4,"[bilstm, cnn_1d, gru, mlp_current_day]"
2,no_high_sleep_session,7,4,"[bilstm, cnn_1d, gru, mlp_current_day]"
3,no_high_sleep_session,14,4,"[bilstm, cnn_1d, gru, mlp_current_day]"
